# Welcome to PYNQ Audio
This notebook shows the basic recording and playback features of the AUP-ZU3.  
It uses the audio jack `HP+MIC` to play back recordings; it can take inputs from 
the microphone on `HP+MIC` or `LINE_IN`. Pre-recorded audio sample can also be taken
as input. Moreover, visualization with matplotlib is shown.

The notebook works with **both** builds of the AIC3104 interface. It reads the
design's version register (`REG_VER`) and automatically uses the matching
capture/playback path:

* **version 1 &rarr; polling build** (`aic3104_poll.sv`): the CPU moves one
  sample at a time through the RX/TX FIFOs.
* **version 2 &rarr; DMA build** (`aic3104_dma.sv`): the `axi_dma_writer`/`axi_dma_reader`
  engines stream whole DDR buffers at hardware rate.

## Create new audio object

In [ ]:
from pynq import Overlay, allocate
import time
from aic3104 import AIC3104
import numpy as np
from scipy.io import wavfile
import asyncio
import wave

# 1. Load the Overlay
ol = Overlay("hw_wrapper.bit")

# 2. Check IP Dictionary
print("Available IPs:", ol.ip_dict.keys())

# release audio reset
rst = ol.audio.axi_gpio_0.channel1 
rst.write(0x1, 0x1)

codec = AIC3104(ol.audio.axi_iic_0)
print("on the bus:", codec.scan())          # should now list 0x18
codec.reset_controller()            # clear a wedged AXI-IIC core if needed
print("on the bus:", codec.scan())          # should now list 0x18
codec.init(mclk_hz=12_288_000, fs=48000, word_len=16)     # set mclk_hz to YOUR design's MCLK
codec.select_microphone()
print(codec.dump())

# Locate the AIC3104 audio control IP. The polling and DMA builds package
# different wrappers (e.g. aic3104_wrapper_0 vs aic3104_dma_wrapper_0), but both
# names contain "aic3104", so find it generically instead of hard-coding one.
aic_name = next(n.split('/')[-1] for n in ol.ip_dict if 'aic3104' in n.lower())
aic = getattr(ol.audio, aic_name)
print("Audio control IP:", aic_name)

In [ ]:
# ---------------------------------------------------------------------------
# Register maps for BOTH builds of the AIC3104 interface.
#
# REG_DEV / REG_VER / REG_INT sit at the SAME addresses in both builds, so we
# always read REG_VER first and then pick the right register set:
#     version 1 -> polling build (aic3104_poll.sv): one sample per AXI access
#     version 2 -> DMA build      (aic3104_dma.sv):  buffer/address DMA
# ---------------------------------------------------------------------------

# -- common (identical in both builds) --
REG_DEV = 0x200
REG_VER = 0x204
REG_INT = 0x208

# -- DMA build (version 2) --
REG_TX_ADDR_LO    = 0x0
REG_TX_ADDR_HI    = 0x4
REG_TX_BYTES      = 0x8
REG_TX_START      = 0xc
REG_TX_BYTES_READ = 0x10
REG_TX_STAT       = 0x14
REG_RX_ADDR_LO    = 0x100
REG_RX_ADDR_HI    = 0x104
REG_RX_BYTES      = 0x108
REG_RX_START      = 0x10c
REG_RX_BYTES_READ = 0x110
REG_RX_STAT       = 0x114
DONE_BIT          = 1 << 31
UNDERFLOW_BIT     = 1 << 3        # REG_TX_STAT[3] = playback underflow (TX starved)

# -- polling build (version 1) --
P_TX_CTRL  = 0x0
P_TX_COUNT = 0x4
P_TX_DATA  = 0x8
P_TX_STAT  = 0xc          # bit0 = tx_full
P_RX_CTRL  = 0x110
P_RX_COUNT = 0x114
P_RX_DATA  = 0x118
P_RX_STAT  = 0x11c        # bit0 = rx_empty

# ---- Identify the build by reading the version register --------------------
device  = aic.read(REG_DEV)
version = aic.read(REG_VER)
dev_str = device.to_bytes(4, "big").decode("ascii", "replace")
IS_DMA  = (version == 2)
mode    = "DMA" if IS_DMA else "polling"
print(f"I2S Device = {dev_str!r} (0x{device:08x}), Version = {version} -> {mode} build")

# ===========================================================================
#  Capture (record) -- fill a buffer of packed 32-bit L/R samples
# ===========================================================================
def dma_capture(buf):
    """version 2: hand the engine a buffer address + length, then poll done."""
    aic.write(REG_RX_ADDR_LO, buf.device_address & 0xFFFFFFFF)
    aic.write(REG_RX_ADDR_HI, buf.device_address >> 32)   # addr can exceed 32 bits
    aic.write(REG_RX_BYTES,   buf.nbytes)
    aic.write(REG_RX_START,   1)                          # cfg_start pulse
    while not (aic.read(REG_RX_STAT) & DONE_BIT):
        pass
    return aic.read(REG_RX_BYTES_READ)

def poll_capture(buf):
    """version 1: read one sample at a time out of the RX FIFO.

    The CPU cannot poll a 48 kHz FIFO from Python without falling behind, so
    samples are dropped and the recording is a slow, lossy grab -- exactly the
    limitation the DMA build removes. We wait for rx_empty to clear before each
    read so we never latch a stale word."""
    n = buf.shape[0]
    aic.write(P_RX_CTRL, 1)                            # enable RX capture
    for i in range(n):
        while aic.read(P_RX_STAT) & 1:                 # wait while rx_empty
            pass
        buf[i] = aic.read(P_RX_DATA)                   # reading pops one sample
    aic.write(P_RX_CTRL, 0)                            # stop capture
    return n * 4

def audio_capture(buf):
    """Record into buf using whichever path the loaded build provides."""
    return dma_capture(buf) if IS_DMA else poll_capture(buf)

# ===========================================================================
#  Playback -- stream a buffer of packed 32-bit L/R samples out the codec
# ===========================================================================
def dma_play(buf):
    """version 2: hand the reader (MM2S) a buffer address + length."""
    aic.write(REG_TX_ADDR_LO, buf.device_address & 0xFFFFFFFF)
    aic.write(REG_TX_ADDR_HI, buf.device_address >> 32)
    aic.write(REG_TX_BYTES,   buf.nbytes)
    aic.write(REG_TX_START,   1)                          # cfg_start pulse
    # done (bit 31) is sticky from the previous run: wait for it to clear (the
    # reader took the new transfer) and then set again (this transfer finished).
    while aic.read(REG_TX_STAT) & DONE_BIT:               # stale done clears
        pass
    while not (aic.read(REG_TX_STAT) & DONE_BIT):         # new transfer done
        pass
    if aic.read(REG_TX_STAT) & UNDERFLOW_BIT:
        print("WARNING: playback underflow - reader could not keep the TX FIFO fed")
    return aic.read(REG_TX_BYTES_READ)

def poll_play(buf):
    """version 1: push one sample at a time into the TX FIFO, throttling on
    tx_full. As with poll_capture, Python can't sustain 48 kHz, so playback
    will underflow (audible gaps) -- the DMA build is what fixes this."""
    n = buf.shape[0]
    for i in range(n):
        while aic.read(P_TX_STAT) & 1:                 # wait while tx_full
            pass
        aic.write(P_TX_DATA, int(buf[i]))
    return n * 4

def audio_playback(buf):
    """Play buf out the codec using whichever path the loaded build provides."""
    return dma_play(buf) if IS_DMA else poll_play(buf)

In [ ]:
print("Starting Capture")
# The DMA build records in real time; the polling build can only manage a short,
# lossy grab from Python, so keep the duration small when version == 1.
SAMPLE_RATE = 48000          # set this to your I2S/codec rate (44100, 48000, ...)
TIME = 5 if IS_DMA else 1    # seconds  (polling is slow -- keep it short)
N = TIME * SAMPLE_RATE
buf = allocate(shape=(N,), dtype=np.uint32)   # default: non-cacheable -> HP port

read_bytes = audio_capture(buf)
print(f"Capture Complete! ({read_bytes} bytes, {read_bytes // 4} frames)")

# --- Unpack the 32-bit word into two signed 16-bit samples ---
# Left in the upper 16 bits, right in the lower 16 bits (same packing in both builds).
left  = (buf >> 16).astype(np.uint16).view(np.int16)
right = (buf & 0xFFFF).astype(np.uint16).view(np.int16)

# --- Interleave L R L R for a stereo WAV ---
stereo = np.empty(N * 2, dtype=np.int16)
stereo[0::2] = left
stereo[1::2] = right

# --- Write the WAV ---
with wave.open("capture.wav", "wb") as w:
    w.setnchannels(2)
    w.setsampwidth(2)          # 2 bytes = 16 bits
    w.setframerate(SAMPLE_RATE)
    w.writeframes(stereo.tobytes())

## Play back the capture
Stream the captured buffer back out through the codec. `audio_playback()` picks
the path that matches the build detected from `REG_VER`:

* **DMA build (version 2):** the `axi_dma_reader` (MM2S) engine walks the DDR
  buffer and feeds the I2S TX FIFO, which the DAC drains one packed L/R sample
  per 48&nbsp;kHz frame.
* **Polling build (version 1):** the CPU pushes each sample into the TX FIFO,
  throttling on `tx_full` (slow from Python, so expect underflow gaps).

Audio comes out of the **LINE OUT** jack (green, J4). The capture buffer `buf`
is already in the exact 32-bit packing the TX path expects
(`bits[31:16]=left`, `bits[15:0]=right` &mdash; the same words the capture path
produced), so we replay it **verbatim**: no WAV round-trip and no L/R re-packing.

> **Note on the DMA done bit.** `REG_TX_STAT[31]` (done) is *sticky*: it stays
> set from the previous playback until the next `START` clears it. So the DMA
> path waits for it to **drop** (the reader accepted the new transfer) and then
> **rise** again (this transfer finished). Polling only for "set" would fall
> through instantly on a re-run, reading back a stale completion.

In [ ]:
# --- Playback (works for both builds) ---
# Replay the just-captured buffer back out through the codec. `buf` already holds
# the 32-bit packed L/R words the capture produced, so we stream it back verbatim
# -- no WAV round-trip, no re-packing, so L/R can't get swapped. audio_playback()
# dispatches to the DMA reader (version 2) or the polling TX FIFO loop (version 1)
# based on the version we read earlier. The DAC drives LINE OUT (green jack J4).

print(f"Playing {N/SAMPLE_RATE:.2f} s ({N} frames) out LINE OUT via the {mode} path ...")
played = audio_playback(buf)
print(f"Played {played} bytes ({played // 4} frames); playback complete!")

## Play in notebook
Since the samples are in 24-bit PCM format, 
users can play the audio directly in notebook.

In [ ]:
from IPython.display import Audio as IPAudio
IPAudio("capture.wav")

## Plotting PCM data

Users can display the audio data in notebook:

1. Plot the audio signal's amplitude over time.
2. Plot the spectrogram of the audio signal.

The next cell reads the saved audio file and processes it into a `numpy` array.
Note that if the audio sample width is not standard, additional processing
is required. In the following example, the `sample_width` is read from the
wave file itself (24-bit dual-channel PCM audio, where `sample_width` is 3 bytes).

In [ ]:
%matplotlib inline
import wave
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.fftpack import fft

wav_path = "capture.wav"
with wave.open(wav_path, 'r') as wav_file:
    raw_frames = wav_file.readframes(-1)
    num_frames = wav_file.getnframes()
    num_channels = wav_file.getnchannels()
    sample_rate = wav_file.getframerate()
    sample_width = wav_file.getsampwidth()
    
temp_buffer = np.empty((num_frames, num_channels, 4), dtype=np.uint8)
raw_bytes = np.frombuffer(raw_frames, dtype=np.uint8)
temp_buffer[:, :, :sample_width] = raw_bytes.reshape(-1, num_channels, 
                                                    sample_width)
temp_buffer[:, :, sample_width:] = \
    (temp_buffer[:, :, sample_width-1:sample_width] >> 7) * 255
frames = temp_buffer.view('<i4').reshape(temp_buffer.shape[:-1])

### 1. Amplitude over time

In [ ]:
for channel_index in range(num_channels):
    plt.figure(num=None, figsize=(15, 3))
    plt.title('Audio in Time Domain (Channel {})'.format(channel_index))
    plt.xlabel('Time in s')
    plt.ylabel('Amplitude')
    time_axis = np.arange(0, num_frames/sample_rate, 1/sample_rate)
    plt.plot(time_axis, frames[:, channel_index])
    plt.show()

### 2. Frequency spectrum

In [ ]:
for channel_index in range(num_channels):
    plt.figure(num=None, figsize=(15, 3))
    plt.title('Audio in Frequency Demain (Channel {})'.format(channel_index))
    plt.xlabel('Frequency in Hz')
    plt.ylabel('Magnitude')
    temp = fft(frames[:, channel_index])
    yf = temp[1:len(temp)//2]
    xf = np.linspace(0.0, sample_rate/2, len(yf))
    plt.plot(xf, abs(yf))
    plt.show()

### 3. Frequency spectrum over time
Use the `classic` plot style for better display.

In [ ]:
for channel_index in range(num_channels):
    np.seterr(divide='ignore', invalid='ignore')
    matplotlib.style.use("classic")
    plt.figure(num=None, figsize=(15, 3))
    plt.title('Signal Spectogram (Channel {})'.format(channel_index))
    plt.xlabel('Time in s')
    plt.ylabel('Frequency in Hz')
    plt.specgram(frames[:, channel_index], Fs=sample_rate)

Copyright (c) 2022 Xilinx, Inc 
<br>
Copyright (C) 2022-2025 Advanced Micro Devices, Inc. 
<br>
SPDX-License-Identifier: BSD-3-Clause 